In [1]:
# ============================================================
# CODEALPHA AI INTERNSHIP - TASK 2
# CHATBOT FOR FAQs
# ============================================================

# Install required libraries
!pip install -q gradio scikit-learn nltk

# ------------------------------------------------------------
# Import libraries
# ------------------------------------------------------------

import re
import nltk
import gradio as gr

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK stopwords
nltk.download("stopwords", quiet=True)

# ------------------------------------------------------------
# FAQ DATASET
# Topic: AI / Machine Learning
# ------------------------------------------------------------

faq_data = [
    {
        "question": "What is Artificial Intelligence?",
        "answer": "Artificial Intelligence (AI) is a branch of computer science that enables machines to perform tasks that normally require human intelligence, such as learning, reasoning, problem-solving and decision-making."
    },
    {
        "question": "What is Machine Learning?",
        "answer": "Machine Learning is a subset of Artificial Intelligence in which computers learn patterns from data and use those patterns to make predictions or decisions without being explicitly programmed for every task."
    },
    {
        "question": "What are the types of Machine Learning?",
        "answer": "The main types of Machine Learning are supervised learning, unsupervised learning and reinforcement learning."
    },
    {
        "question": "What is supervised learning?",
        "answer": "Supervised learning is a Machine Learning method where a model learns from labelled training data. Examples include classification and regression."
    },
    {
        "question": "What is unsupervised learning?",
        "answer": "Unsupervised learning works with data that does not have labelled outputs. It is commonly used for finding patterns, groups and structures in data. Clustering is a common example."
    },
    {
        "question": "What is reinforcement learning?",
        "answer": "Reinforcement learning is a Machine Learning technique in which an agent learns by interacting with an environment and receiving rewards or penalties for its actions."
    },
    {
        "question": "What is Deep Learning?",
        "answer": "Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers to learn complex patterns from large amounts of data."
    },
    {
        "question": "What is a neural network?",
        "answer": "A neural network is a computing model inspired by the human brain. It consists of interconnected nodes called neurons that process information and learn patterns from data."
    },
    {
        "question": "What is Python?",
        "answer": "Python is a high-level, general-purpose programming language widely used in Artificial Intelligence, Machine Learning, data science, web development and automation."
    },
    {
        "question": "What is NLP?",
        "answer": "Natural Language Processing (NLP) is a field of Artificial Intelligence that enables computers to understand, process and generate human language."
    },
    {
        "question": "What is data preprocessing?",
        "answer": "Data preprocessing is the process of preparing raw data for Machine Learning. It can include cleaning data, handling missing values, removing unnecessary information and transforming data into a suitable format."
    },
    {
        "question": "What is classification?",
        "answer": "Classification is a supervised Machine Learning technique used to assign data to predefined categories or classes."
    },
    {
        "question": "What is regression?",
        "answer": "Regression is a supervised Machine Learning technique used to predict continuous numerical values, such as price, temperature or sales."
    },
    {
        "question": "What is clustering?",
        "answer": "Clustering is an unsupervised learning technique that groups similar data points together without using predefined labels."
    },
    {
        "question": "What is an AI chatbot?",
        "answer": "An AI chatbot is a software application that interacts with users through natural language and provides responses based on its knowledge, rules or trained models."
    }
]

# ------------------------------------------------------------
# Extract questions and answers
# ------------------------------------------------------------

questions = [item["question"] for item in faq_data]
answers = [item["answer"] for item in faq_data]

# ------------------------------------------------------------
# NLP PREPROCESSING
# ------------------------------------------------------------

stop_words = set(stopwords.words("english"))


def preprocess_text(text):
    """
    Clean and preprocess text using basic NLP techniques.
    """

    # Convert to lowercase
    text = text.lower()

    # Remove special characters and numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Join words again
    return " ".join(words)


# ------------------------------------------------------------
# Preprocess FAQ questions
# ------------------------------------------------------------

processed_questions = [
    preprocess_text(question)
    for question in questions
]

# ------------------------------------------------------------
# TF-IDF VECTORIZER
# ------------------------------------------------------------

vectorizer = TfidfVectorizer()

faq_vectors = vectorizer.fit_transform(
    processed_questions
)


# ------------------------------------------------------------
# CHATBOT FUNCTION
# ------------------------------------------------------------

def chatbot_response(user_question):

    # Check empty input
    if user_question is None or user_question.strip() == "":
        return "Please enter a question."

    # Preprocess user question
    processed_user_question = preprocess_text(
        user_question
    )

    # Convert user question into TF-IDF vector
    user_vector = vectorizer.transform(
        [processed_user_question]
    )

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        faq_vectors
    )[0]

    # Find best matching FAQ
    best_match_index = similarity_scores.argmax()

    best_score = similarity_scores[best_match_index]

    # Minimum similarity threshold
    threshold = 0.15

    if best_score < threshold:
        return (
            "Sorry, I don't have an answer for that question. "
            "Please ask something related to Artificial Intelligence "
            "or Machine Learning."
        )

    # Return matching answer
    return answers[best_match_index]


# ------------------------------------------------------------
# CLEAR CHAT FUNCTION
# ------------------------------------------------------------

def clear_chat():
    return ""


# ============================================================
# GRADIO CHAT INTERFACE
# ============================================================

with gr.Blocks(
    title="AI FAQ Chatbot"
) as demo:

    gr.Markdown(
        """
        # 🤖 AI & Machine Learning FAQ Chatbot

        ### CodeAlpha Artificial Intelligence Internship - Task 2

        Ask questions about Artificial Intelligence and
        Machine Learning.
        """
    )

    gr.Markdown(
        """
        **Examples:**
        - What is Artificial Intelligence?
        - What is Machine Learning?
        - What are the types of Machine Learning?
        - What is supervised learning?
        - What is Deep Learning?
        - What is NLP?
        - What is Python?
        """
    )

    # --------------------------------------------------------
    # Chatbot interface
    # --------------------------------------------------------

    chatbot = gr.Chatbot(
        label="AI FAQ Assistant",
        height=400
    )

    user_input = gr.Textbox(
        label="Ask your question",
        placeholder="Type your question here...",
        lines=2
    )

    with gr.Row():

        send_button = gr.Button(
            "💬 Ask",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    # --------------------------------------------------------
    # Send message function
    # --------------------------------------------------------

    def respond(message, history):

        if not message.strip():
            return history

        answer = chatbot_response(message)

        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": answer}
        ]

        return history

    # Send button
    send_button.click(
        fn=respond,
        inputs=[
            user_input,
            chatbot
        ],
        outputs=[
            chatbot
        ]
    ).then(
        fn=lambda: "",
        inputs=[],
        outputs=[user_input]
    )

    # Enter key
    user_input.submit(
        fn=respond,
        inputs=[
            user_input,
            chatbot
        ],
        outputs=[
            chatbot
        ]
    ).then(
        fn=lambda: "",
        inputs=[],
        outputs=[user_input]
    )

    # Clear button
    clear_button.click(
        fn=lambda: ([], ""),
        inputs=[],
        outputs=[
            chatbot,
            user_input
        ]
    )


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d90cd02d8787fa2dd7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
